# BESS Layout Optimization Engine

Standalone engineering notebook. All layout, sizing and routing logic lives in the shared `core` package (no dependency on the Streamlit app); rendering lives in `viz`. The app and this notebook call the **same** `build_config` + engine, so results match.


In [ ]:
import sys, os
# Make the repo root importable when running from the notebook/ folder.
sys.path.insert(0, os.path.abspath('..'))

from core import build_config, run_bess_optimization, run_colocated_optimization, size_system
from viz import print_comparison, plot_comparison, plot_all_standalone

# =========================================================
# CONFIGURATION  (single source of truth via build_config)
# =========================================================
cable_corridor = [(15.4, 0), (21.9, 0), (47.7, 90.4), (31.9, 90.4)]
out_of_scope   = [(21.9, 16), (53.3, 16), (53.3, 90.4), (47.7, 90.4)]

CONFIG = build_config(
    site_vertices=[
        (0, 0), (53.3, 0), (53.3, 16), (21.9, 16),
        (47.7, 90.4), (8, 90.4), (0, 90.4),
    ],
    non_buildable=[cable_corridor],
    restricted=[out_of_scope],
    max_bess_per_mvs=4,
    max_cable_length=25,
    mvs_scoring_radius=25,
    min_mvs_spacing=0,
    grid_resolution=2.0,
    bess_unit_mwh=5.0,
    mvs_station_mw=2.5,
)

In [ ]:
import time

scenarios = []
for mode in ("conservative", "aggressive", "ultra_aggressive", "hyper_pack"):
    t0 = time.perf_counter()
    res = run_bess_optimization(CONFIG, mode=mode, verbose=False)
    elapsed = time.perf_counter() - t0
    print(f"{mode:<20s} -> {res['metrics']['bess_count']:3d} BESS in {elapsed:5.1f} s")
    scenarios.append(res)

# Metrics table (note: CONFIG is passed so all 4 scenarios are shown and sized).
print_comparison(CONFIG, *scenarios)

In [ ]:
# System sizing per scenario: total MW / MWh and 2H / 3H / 4H duration class.
for res in scenarios:
    m = res['metrics']
    s = size_system(
        m['bess_count'], m['mvs_count'],
        CONFIG['bess_unit_mwh'], CONFIG['mvs_station_mw'],
    )
    print(f"{res['mode']:<18s} {s['total_mw']:6.1f} MW | {s['total_mwh']:7.1f} MWh | "
          f"{s['duration_h']:.2f} h -> {s['duration_label']}")

In [ ]:
# 2x2 panel overview
plot_comparison(*scenarios, config=CONFIG)

In [ ]:
# Full-size standalone figures
plot_all_standalone(*scenarios, config=CONFIG)

In [ ]:
# Optional: co-located / paired MVS scenario (shared-pad hubs)
CO_CONFIG = build_config(
    site_vertices=CONFIG['site_vertices'],
    non_buildable=[cable_corridor],
    restricted=[out_of_scope],
    max_bess_per_mvs=4,
    colocation={'enabled': True, 'group_size': 2, 'pad_gap': 0.5},
)
co_res = run_colocated_optimization(CO_CONFIG, mode='aggressive', verbose=True)